# Geometry-V1 operational handoff
PREPARED_NOT_EXECUTED; transport-only; science_denominator=0.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
import json, os, pathlib, re, shutil, subprocess, sys
import numpy as np
from PIL import Image
from google.colab import userdata
from datetime import datetime, timezone
REPO_URL='https://github.com/RICHAAARC/CEG-WM.git'; BRANCH='Geometry-V1'
DRIVE_ROOT=pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/Batch2B'); MAX_CONTROL_BYTES=1024; MAX_RECEIPT_BYTES=262144; MAX_ARCHIVE_BYTES=524288; MAX_SIDECAR_BYTES=256
repo=pathlib.Path('/content/geometry-v1-source'); input_dir=pathlib.Path('/content/geometry-v1-inputs'); root_key=''; hf_token=''; runner_env=None; process=None; control_read=None; control_write=None; input_paths=[]; fixed_image=None; fixed_array=None
def build_fixed_operational_rgb() -> Image.Image:
 yy,xx=np.indices((512,512),dtype=np.uint16); array=np.stack(((3*xx+5*yy)%256,(7*xx+2*yy+17*(xx//32))%256,(xx^(3*yy))%256),axis=-1).astype(np.uint8); array[40:180,55:225]=(241,67,31); array[300:465,335:493]=(19,181,223); return Image.fromarray(array)
def checkout_identity():
 commit=subprocess.run(['git','rev-parse','HEAD'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip(); branch=subprocess.run(['git','branch','--show-current'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip(); clean=subprocess.run(['git','status','--porcelain'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip()
 if re.fullmatch(r'[0-9a-f]{40}',commit) is None or branch!=BRANCH or clean: raise RuntimeError('checkout identity differs')
 return commit
def parse_child(rc, control_read, run_id):
 line=os.read(control_read,MAX_CONTROL_BYTES+1)
 if len(line)>MAX_CONTROL_BYTES: raise RuntimeError('bounded control exceeded')
 text=line.decode('utf-8','strict'); success='CEGWM_GEOMETRY_V1_OPERATIONAL_PREFLIGHT '; failure='CEGWM_GEOMETRY_V1_OPERATIONAL_FAILURE '
 if not text.endswith('\n') or len(text.splitlines())!=1: raise RuntimeError('exactly one control line required')
 if text.startswith(success): prefix,status=success,'success'
 elif text.startswith(failure): prefix,status=failure,'failure'
 else: raise RuntimeError('invalid control prefix')
 try: payload=json.loads(text[len(prefix):])
 except json.JSONDecodeError as error: raise RuntimeError('invalid control JSON') from error
 unavailable={'status','underlying_status','artifact_status','failure_point','run_id'}; complete={'status','run_id','artifact_status','archive_filename','sidecar_filename','receipt_bytes','receipt_sha256','archive_bytes'}; allowed=complete if status=='success' else complete|{'underlying_status','failure_point'}
 if status=='failure' and payload.get('artifact_status')=='unavailable':
  if set(payload)!=unavailable or payload.get('underlying_status') not in {'unknown','operational_failure'}: raise RuntimeError('invalid artifact-unavailable control')
 elif set(payload)!=allowed or payload.get('artifact_status')!='complete' or (status=='failure' and payload.get('underlying_status')!='operational_failure'): raise RuntimeError('invalid packaged control')
 if payload.get('status')!=status or payload.get('run_id')!=run_id or (rc==0)!=(status=='success'): raise RuntimeError('control/return-code mismatch')
 if payload.get('artifact_status')=='complete' and (not isinstance(payload['receipt_bytes'],int) or not 0<=payload['receipt_bytes']<=MAX_RECEIPT_BYTES or not isinstance(payload['archive_bytes'],int) or not 0<=payload['archive_bytes']<=MAX_ARCHIVE_BYTES or re.fullmatch(r'[0-9a-f]{64}',payload['receipt_sha256']) is None or payload['archive_filename']!=run_id+'.zip' or payload['sidecar_filename']!=run_id+'.zip.sha256'): raise RuntimeError('invalid package declaration')
 return status,payload
try:
 if repo.exists() or input_dir.exists(): raise FileExistsError('local create-only path exists')
 subprocess.run(['git','clone','--single-branch','--branch',BRANCH,REPO_URL,str(repo)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
 checkout_commit=checkout_identity(); RUN_ID='geometry-v1-b2b-'+checkout_commit[:12]+'-operational-01'; run_dir=DRIVE_ROOT/('Geometry-V1-'+checkout_commit[:12]+'-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
 if run_dir.exists(): raise FileExistsError('create-only Drive target exists')
 subprocess.run([sys.executable,'-m','pip','install',str(repo)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); checkout_identity()
 input_dir.mkdir(); fixed_image=build_fixed_operational_rgb(); fixed_array=np.asarray(fixed_image); fixed_path=input_dir/'geometry_v1_batch2b_fixed_rgb.png'; input_paths.append(fixed_path)
 if fixed_image.mode!='RGB' or fixed_image.size!=(512,512) or fixed_array.dtype!=np.uint8 or fixed_array.shape!=(512,512,3): raise RuntimeError('fixed RGB input identity differs')
 with fixed_path.open('xb') as handle: fixed_image.save(handle,format='PNG')
 root_key=userdata.get('CEG_WM_ROOT_KEY'); hf_token=userdata.get('HF_TOKEN')
 if not root_key or not hf_token: raise RuntimeError('required secret unavailable')
 runner_env={n:v for n,v in os.environ.items() if all(x not in n.upper() for x in ('TOKEN','KEY','SECRET'))}; runner_env['HF_TOKEN']=hf_token; runner_env['CEG_WM_ROOT_KEY']=root_key; control_read,control_write=os.pipe()
 command=[sys.executable,'-m','experiments.run_geometry_v1_qk_operational_preflight','--repo-root',str(repo),'--expected-exact',checkout_commit,'--output-root',str(run_dir),'--control-fd',str(control_write),str(fixed_path)]
 process=subprocess.Popen(command,cwd=repo,env=runner_env,pass_fds=(control_write,),stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); os.close(control_write); control_write=None; process.wait(timeout=1800); status,payload=parse_child(process.returncode,control_read,RUN_ID)
 if status=='failure':
  if payload['artifact_status']=='unavailable': raise RuntimeError('child transport artifact unavailable; Drive evidence retained')
  raise RuntimeError('child reported operational failure; Drive evidence retained')
finally:
 root_key=''; hf_token=''; fixed_image=None; fixed_array=None
 if runner_env is not None: runner_env.pop('HF_TOKEN',None); runner_env.pop('CEG_WM_ROOT_KEY',None)
 if process is not None and process.poll() is None: process.kill(); process.wait()
 for fd in (control_read,control_write):
  if fd is not None: os.close(fd)
 for path in input_paths:
  if path.exists(): path.unlink()
 if input_dir.exists(): input_dir.rmdir()
 if repo.exists(): shutil.rmtree(repo)


In [ ]:
# Artifact-only inspection: this cell never starts a runner.
if 'run_dir' not in globals() or not run_dir.is_dir(): raise RuntimeError('retained Drive target unavailable')
if 'payload' in globals() and payload.get('artifact_status')=='unavailable':
 print({'run_id':RUN_ID,'operational_status':payload['underlying_status'],'artifact_status':'unavailable','science_denominator':0}); raise RuntimeError('artifact unavailable; no terminal pair')
status_files=[p for p in (run_dir/'success.json',run_dir/'failure.json') if p.is_file()]; checkpoint=run_dir/'checkpoint.json'; archive=run_dir/(RUN_ID+'.zip'); sidecar=run_dir/(RUN_ID+'.zip.sha256')
if len(status_files)!=1 or not checkpoint.is_file() or not archive.is_file() or not sidecar.is_file() or archive.stat().st_size>MAX_ARCHIVE_BYTES: raise RuntimeError('terminal artifact declaration differs')
sidecar_value=sidecar.read_bytes()
if len(sidecar_value)>MAX_SIDECAR_BYTES or re.fullmatch(rb'[0-9a-f]{64}  '+re.escape(archive.name.encode())+rb'\n',sidecar_value) is None: raise RuntimeError('terminal sidecar differs')
print({'run_id':RUN_ID,'operational_status':status_files[0].stem,'artifact_status':'complete','science_denominator':0})
